In [1]:
import tarfile
import os
import numpy as np
from pathlib import Path
from itertools import chain


In [2]:
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if ((p / "ebm_nlp_2_00").exists())), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/ebm_nlp_2_00 from the current notebook directory")

RAW_DATA_CANDIDATES = [
    PROJECT_ROOT / "ebm_nlp_2_00",
]
DATA_DIR = next((p for p in RAW_DATA_CANDIDATES if (p / "documents").exists() and (p / "annotations").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find raw EBM-NLP documents/annotations folders")

print(f"Using raw data from: {DATA_DIR}")

Using raw data from: Y:\222\ebm_nlp_2_00


In [3]:
def get_doc_ids(split="train", label_type="participants"):


    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type 
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
    print(f"{label_type} {split}: {len(doc_ids)} docs from {train_dir}")
    return sorted(doc_ids)

In [4]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

In [5]:
def load_labels_for_doc(doc_id, label_type="participants", split="train"):

    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

In [6]:
def hierarchical_to_bio(tags):


    bio = []
    prev = 0

    for t in tags:
        t = int(t)
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")

        prev = t
        

    return bio

def convert_all_labels_to_bio(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_bio(doc_labels)
    return labels

## Load the Data

In [7]:

p_ids = set(get_doc_ids("train", "participants"))
i_ids = set(get_doc_ids("train", "interventions"))
o_ids = set(get_doc_ids("train", "outcomes"))
train_ids = sorted(list(p_ids & i_ids & o_ids))

p_test_ids = set(get_doc_ids("test", "participants"))
i_test_ids = set(get_doc_ids("test", "interventions"))
o_test_ids = set(get_doc_ids("test", "outcomes"))
test_ids = sorted(list(p_test_ids & i_test_ids & o_test_ids))

train_tokens = load_documents(train_ids)
test_tokens = load_documents(test_ids)

train_labels_p = load_labels(train_ids, "participants", "train")
train_labels_i = load_labels(train_ids, "interventions", "train")
train_labels_o = load_labels(train_ids, "outcomes", "train")

test_labels_p = load_labels(test_ids, "participants", "test")
test_labels_i = load_labels(test_ids, "interventions", "test")
test_labels_o = load_labels(test_ids, "outcomes", "test")

train_labels_p = convert_all_labels_to_bio(train_labels_p)
test_labels_p = convert_all_labels_to_bio(test_labels_p)

train_labels_i = convert_all_labels_to_bio(train_labels_i)
test_labels_i = convert_all_labels_to_bio(test_labels_i)

train_labels_o = convert_all_labels_to_bio(train_labels_o)
test_labels_o = convert_all_labels_to_bio(test_labels_o)

participants train: 4609 docs from Y:\222\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\participants\train
interventions train: 4746 docs from Y:\222\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\interventions\train
outcomes train: 4681 docs from Y:\222\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\outcomes\train
participants test/gold: 189 docs from Y:\222\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\participants\test\gold
interventions test/gold: 187 docs from Y:\222\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\interventions\test\gold
outcomes test/gold: 190 docs from Y:\222\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\outcomes\test\gold


In [8]:
import numpy as np
import nltk
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer


nltk.download("punkt")
nltk.download("punkt_tab")

processed_dir = PROJECT_ROOT / "ebm_nlp_2_00" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def split_tokens_with_nltk(tokens):

    text = " ".join(tokens)
    sentences = nltk.sent_tokenize(text)

    spans = []
    cursor = 0
    token_idx = 0

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        sent_tokens = sent.split()
        start_idx = token_idx
        end_idx = start_idx + len(sent_tokens)

        spans.append({
            "text": sent,
            "start": start_idx,
            "end": end_idx
        })

        token_idx = end_idx
        cursor += len(sent)

    return spans

def vectorize_and_label_sentences_nltk(doc_ids, tokens_nested, labels_p_nested, labels_i_nested, labels_o_nested, split_name="Data"):
    sent_texts = []
    sent_labels_p = []
    sent_labels_i = []
    sent_labels_o = []
    sent_doc_ids = []

    print(f"Processing {split_name} into sentences with NLTK...")

    for doc_id, tokens, lp, li, lo in tqdm(
        zip(doc_ids, tokens_nested, labels_p_nested, labels_i_nested, labels_o_nested),
        total=len(tokens_nested)
    ):
        sent_spans = split_tokens_with_nltk(tokens)

        for span in sent_spans:
            sent_text = span["text"]
            start_idx = span["start"]
            end_idx = span["end"]

            if len(sent_text.split()) < 3:
                continue

            sent_texts.append(sent_text)
            sent_doc_ids.append(doc_id)

            def has_entity(label_list_doc, start_idx, end_idx):
                segment = label_list_doc[start_idx:end_idx]
                return 1 if any(lab != "O" for lab in segment) else 0

            sent_labels_p.append(has_entity(lp, start_idx, end_idx))
            sent_labels_i.append(has_entity(li, start_idx, end_idx))
            sent_labels_o.append(has_entity(lo, start_idx, end_idx))

    sent_vectors = embedder.encode(sent_texts, show_progress_bar=True)

    return (
        np.array(sent_vectors),
        np.array(sent_labels_p),
        np.array(sent_labels_i),
        np.array(sent_labels_o),
        sent_texts,
        sent_doc_ids
    )

X_train_sent, y_p_train_sent, y_i_train_sent, y_o_train_sent, train_sent_texts, train_sent_doc_ids = \
    vectorize_and_label_sentences_nltk(
        train_ids, train_tokens, train_labels_p, train_labels_i, train_labels_o, "Train Split"
    )

X_test_sent, y_p_test_sent, y_i_test_sent, y_o_test_sent, test_sent_texts, test_sent_doc_ids = \
    vectorize_and_label_sentences_nltk(
        test_ids, test_tokens, test_labels_p, test_labels_i, test_labels_o, "Test Split"
    )

np.savez_compressed(
    processed_dir / "sentence_vectors.npz",
    X_train=X_train_sent,
    y_p_train=y_p_train_sent,
    y_i_train=y_i_train_sent,
    y_o_train=y_o_train_sent,
    X_test=X_test_sent,
    y_p_test=y_p_test_sent,
    y_i_test=y_i_test_sent,
    y_o_test=y_o_test_sent,
    texts_train=np.array(train_sent_texts, dtype=object),
    texts_test=np.array(test_sent_texts, dtype=object),
    train_doc_ids=np.array(train_sent_doc_ids, dtype=object),
    test_doc_ids=np.array(test_sent_doc_ids, dtype=object)
)

print(f"Sentence data saved to {processed_dir / 'sentence_vectors.npz'}")
print("X_train shape:", X_train_sent.shape)
print("X_test shape:", X_test_sent.shape)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\eik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\eik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing Train Split into sentences with NLTK...


100%|██████████| 4457/4457 [00:00<00:00, 6376.27it/s]


Batches:   0%|          | 0/1503 [00:00<?, ?it/s]

Processing Test Split into sentences with NLTK...


100%|██████████| 184/184 [00:00<00:00, 6344.77it/s]


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Sentence data saved to Y:\222\ebm_nlp_2_00\processed\sentence_vectors.npz
X_train shape: (48082, 384)
X_test shape: (2020, 384)
